# EDA — anonymised crowdsensed WiFi corpus

Companion notebook for IEEE Latin America Transactions manuscript **ID 11172**.

- **Input:** `../data/datos_wifi_curated_anon.parquet`
- **QC:** `../data/qc_report.json`
- **Figures:** `../reports/figures/` (created on demand)

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA = ROOT / "data" / "datos_wifi_curated_anon.parquet"
QC = ROOT / "data" / "qc_report.json"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

df = pd.read_parquet(DATA)
qc = json.loads(QC.read_text(encoding="utf-8"))
print(
    f"Rows: {len(df):,} | BSSIDs: {df['bssid'].nunique():,} | "
    f"auditors: {df['device_id'].nunique():,}"
)
print("QC curated_rows:", qc.get("curated_rows"), "| input:", qc.get("input"))

## Descriptive statistics

In [ ]:
display(df[["level", "frequency", "estimated_distance", "distance_residual"]].describe())
print("Municipalities:", df["municipality"].nunique())
print("Security classes:\n", df["security_type_parsed"].value_counts().head(10))

## RSSI and heuristic exposure

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["level"], bins=40, ax=ax)
ax.set_xlabel("RSSI (dBm)")
ax.set_title("Received signal strength")
fig.tight_layout()
fig.savefig(FIG / "rssi_hist.png")
plt.show()

if "threat_type_heuristic" in df.columns:
    print(df["threat_type_heuristic"].value_counts())
if "is_anomaly_heuristic" in df.columns:
    print("Heuristic exposure rate:", float(df["is_anomaly_heuristic"].mean()))

## Spatial coverage (rounded coordinates)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["longitude"], df["latitude"], s=2, alpha=0.25)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Auditor locations (3-decimal rounding)")
fig.tight_layout()
fig.savefig(FIG / "spatial_scatter.png")
plt.show()
print("Figures written under:", FIG.as_posix())